# Cookie Cats: Exploratory Data Analysis (EDA) & Data Cleaning

## 1. Introduction

This project analyzes an A/B test conducted in **Cookie Cats**, a popular mobile puzzle game in the "connect-three" genre.

The objective of the experiment is to evaluate the impact of moving the first progression gate from Level 30 to Level 40, and its effect on player retention and engagement.

In casual mobile games, **progression gates** are critical design elements that introduce friction by requiring players to wait or make in-app purchases to continue. 

While this friction can support monetization and long-term engagement, poorly positioned gates may negatively impact the player experience, leading to increased churn.

This analysis aims to determine whether shifting the gate improves or harms key performance indicators, ultimately supporting a data-driven product decision.



## 2. The Dataset

The dataset contains behavioral data from 90,189 players who installed the game during the experiment period.

Each player was randomly assigned to one of two variants:
- **gate_30 (control group):** gate placed at level 30  
- **gate_40 (treatment group):** gate placed at level 40  

### Data Dictionary:

- **userid**: Unique identifier for each player  
- **version**: Experiment group assignment (gate_30 or gate_40)  
- **sum_gamerounds**: Total number of game rounds played within the first 14 days  
- **retention_1**: Whether the player returned 1 day after installation  
- **retention_7**: Whether the player returned 7 days after installation  



## 3. EDA Objectives

Before conducting statistical hypothesis testing, we perform an exploratory data analysis to validate data quality and establish a reliable foundation for the experiment evaluation.

The main objectives of this stage are:

1. **Data Quality Validation**  
   Ensure the dataset is complete, consistent, and free of anomalies that could bias the experiment results.

2. **Player Behavior Analysis**  
   Understand how players interact with the game, particularly in terms of engagement distribution and activity patterns.

3. **Retention Baseline Definition**  
   Establish baseline retention metrics (Day 1 and Day 7) to contextualize the impact of the experiment.

By completing these steps, we ensure that the A/B test results can be interpreted with confidence and translated into actionable product insights.

The final goal is to provide a clear recommendation on whether the new gate position should be adopted in production.

---

### **Setup**

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# Set default Plotly theme
pio.templates.default = 'plotly_white'

# Define a consistent color palette for A/B groups across all visualizations
COLOR_MAP = {'gate_30': '#7C3AED', 'gate_40': '#F59E0B'}

# Read dataset
df = pd.read_csv('../data/raw/cookie_cats.csv')

---

### **Data Quality Validation**

Before proceeding with the experiment analysis, we validate data quality and integrity to ensure that any observed differences between groups can be reliably attributed to the experimental treatment, rather than artifacts introduced by data collection or processing. We begin by verifying the structural integrity of the dataset, including column consistency, and data types

In [2]:
print("--- Dataset Info ---")
print(df.info())
print("\n--- Dataset Shape ---")
print(df.shape)
print("\n--- Missing Values ---")
print(df.isnull().sum())
print("\n--- Dataset Head ---")
df.head()

--- Dataset Info ---
<class 'pandas.DataFrame'>
RangeIndex: 90189 entries, 0 to 90188
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   userid          90189 non-null  int64
 1   version         90189 non-null  str  
 2   sum_gamerounds  90189 non-null  int64
 3   retention_1     90189 non-null  bool 
 4   retention_7     90189 non-null  bool 
dtypes: bool(2), int64(2), str(1)
memory usage: 2.2 MB
None

--- Dataset Shape ---
(90189, 5)

--- Missing Values ---
userid            0
version           0
sum_gamerounds    0
retention_1       0
retention_7       0
dtype: int64

--- Dataset Head ---


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


The absence of missing values suggests a reliable telemetry pipeline, reducing the risk of biased retention estimates due to incomplete user tracking. We must also check for duplicates and for the uniqueness of user identifiers

In [3]:
# Check for duplicate rows
n_duplicate_rows = df.duplicated().sum()
print(f'Duplicate rows: {n_duplicate_rows}')

# Confirm that each userid is unique: one entry per player
n_unique_users = df['userid'].nunique()
print(f'Unique user IDs: {n_unique_users} out of {len(df)} rows')

Duplicate rows: 0
Unique user IDs: 90189 out of 90189 rows


No data quality issues were identified, indicating that the dataset is complete and reliable.

This suggests that:
- Player activity has been consistently tracked
- The telemetry pipeline is stable
- Retention metrics can be interpreted with a lower risk of measurement bias

With data quality validated, we now examine the distribution of total game rounds played (`sum_gamerounds`) to identify extreme player behavior that could influence aggregate metrics.

We also verify that the A/B groups are balanced in size, as a significant imbalance could indicate a randomization failure and undermine the validity of downstream statistical tests

In [4]:
# Confirm group sizes and balance
group_counts = df['version'].value_counts().reset_index()
group_counts.columns = ['version', 'count']
group_counts['share (%)'] = (group_counts['count'] / len(df) * 100).round(2)
print(group_counts)

fig = px.bar(
    group_counts,
    x='version',
    y='count',
    color='version',
    color_discrete_map=COLOR_MAP,
    text='count',
    labels={'version': 'A/B Group', 'count': 'Number of Players'},
    title='Player Count per A/B Group'
)
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, yaxis_range=[0, group_counts['count'].max() * 1.15])
fig.show()

   version  count  share (%)
0  gate_40  45489      50.44
1  gate_30  44700      49.56


The two groups are approximately equal in size (~50/50 split), confirming that randomization was executed correctly. This balance is a prerequisite for valid A/B test comparisons

#### Outlier Detection and Treatment

We analyze the distribution of total game rounds played (`sum_gamerounds`) to identify extreme player behavior.

Outliers in this context may represent:
- Highly engaged users (power players)
- Automated behavior (bots)
- Data anomalies

These extreme values are important because they can disproportionately influence aggregate metrics, such as mean engagement, and potentially distort the interpretation of the experiment results

In [5]:
# Visualize the full distribution to identify extreme values before any filtering
fig = px.box(
    df,
    x='sum_gamerounds',
    color_discrete_sequence=['#7C3AED'],
    labels={'sum_gamerounds': 'Total Game Rounds'},
    title='Distribution of Game Rounds: Full Dataset (Outlier Detection)'
)
fig.show()

In [6]:
# Identify the most extreme value and calculate the 99th percentile threshold
max_rounds = df['sum_gamerounds'].max()
p99 = df['sum_gamerounds'].quantile(0.99)

print(f'Maximum rounds observed: {max_rounds:,}')
print(f'99th percentile: {p99:.0f} rounds')

Maximum rounds observed: 49,854
99th percentile: 493 rounds


A single player with nearly 50,000 rounds was identified: approximately 17x above the 99th percentile. This is a strong candidate for a bot or an automated testing account rather than an organic user.

**Treatment strategy:** We remove only this single extreme observation from the working dataset. All other players, including those at the 99th percentile, are retained to preserve statistical power. The original raw data remains unchanged in `data/raw/`

In [7]:
# Remove the single extreme outlier (the maximum value only)
df = df[df['sum_gamerounds'] < max_rounds].copy()

print(f'Rows removed: 1 (userid with {max_rounds:,} rounds)')
print(f'Remaining rows: {len(df):,}')

fig = px.box(
    df,
    x='sum_gamerounds',
    color_discrete_sequence=['#7C3AED'],
    labels={'sum_gamerounds': 'Total Game Rounds'},
    title='Distribution of Game Rounds: Cleaned Dataset (Without Outliers)'
)
fig.show()

Rows removed: 1 (userid with 49,854 rounds)
Remaining rows: 90,188


---

### **Player Behavior Analysis**

With the dataset validated, we examine the distribution of player engagement (measured by total game rounds played) to understand behavioral patterns across the player base

In [8]:
# Summary statistics for game rounds: overall and per group
print('--- Overall Engagement Metrics ---')
print(df['sum_gamerounds'].describe().round(2))

print('\n--- Engagement Metrics per A/B Group ---')
print(df.groupby('version')['sum_gamerounds'].agg(['count', 'mean', 'median', 'std', 'max']).round(2))

--- Overall Engagement Metrics ---
count    90188.00
mean        51.32
std        102.68
min          0.00
25%          5.00
50%         16.00
75%         51.00
max       2961.00
Name: sum_gamerounds, dtype: float64

--- Engagement Metrics per A/B Group ---
         count   mean  median     std   max
version                                    
gate_30  44699  51.34    17.0  102.06  2961
gate_40  45489  51.30    16.0  103.29  2640


The median (16 rounds) is substantially lower than the mean (~51 rounds), confirming a right-skewed distribution, a common pattern in mobile games where the majority of players engage minimally before churning. This skew makes the median a more robust central tendency measure than the mean for this dataset

In [9]:
# Quantify players who never played a single round (zero-round churn)
# These users installed the game but did not engage at all
non_starters = (df['sum_gamerounds'] == 0).sum()
pct_non_starters = non_starters / len(df) * 100

print(f'Non-starters (0 rounds played): {non_starters:,} ({pct_non_starters:.2f}%)')

Non-starters (0 rounds played): 3,994 (4.43%)


Approximately 4.4% of users never played a single round. This level of zero-round churn is within an expected range for mobile games and does not indicate a systemic onboarding issue. These players are retained in the analysis, as their absence from gameplay is itself a behavioral signal that should be captured in retention metrics

In [10]:
# Histogram of game rounds: capped at 100 to focus on the core player population
# Gate positions are overlaid to contextualize where most players stop relative to the experiment
df_hist = df[df['sum_gamerounds'] <= 100]

fig = px.histogram(
    df_hist,
    x='sum_gamerounds',
    nbins=100,
    color_discrete_sequence=['#7C3AED'],
    labels={'sum_gamerounds': 'Game Rounds Played', 'count': 'Number of Players'},
    title='Distribution of Player Engagement (First 100 Rounds)'
)

# Overlay vertical lines to mark gate positions
fig.add_vline(x=30, line_dash='dash', line_color='#F59E0B',
              annotation_text='Gate 30 (control)', annotation_position='top right')
fig.add_vline(x=40, line_dash='dash', line_color='#10B981',
              annotation_text='Gate 40 (treatment)', annotation_position='top right')

fig.show()

In [11]:
# Side-by-side histograms per A/B group to assess distributional similarity
# Similar shapes confirm that randomization did not introduce behavioral bias between groups
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['gate_30 (control)', 'gate_40 (treatment)'],
                    shared_yaxes=True)

for col, version in enumerate(['gate_30', 'gate_40'], start=1):
    subset = df[(df['version'] == version) & (df['sum_gamerounds'] <= 100)]
    fig.add_trace(
        go.Histogram(x=subset['sum_gamerounds'], nbinsx=100,
                     marker_color=COLOR_MAP[version], name=version),
        row=1, col=col
    )

fig.update_layout(
    title_text='Engagement Distribution by A/B Group (First 100 Rounds)',
    xaxis_title='Game Rounds Played',
    xaxis2_title='Game Rounds Played',
    yaxis_title='Number of Players',
    showlegend=False
)
fig.show()

The distributions are visually similar across both groups, reinforcing confidence in the randomization process. Notably, the majority of players do not reach either gate position (levels 30 or 40), meaning the A/B test primarily captures the behavior of the more engaged segment of the player base

In [12]:
# Quantify the proportion of players who actually reached each gate threshold
total = len(df)
reached_30 = (df['sum_gamerounds'] >= 30).sum()
reached_40 = (df['sum_gamerounds'] >= 40).sum()

print(f'Players who reached Level 30: {reached_30:,} ({reached_30/total*100:.2f}%)')
print(f'Players who reached Level 40: {reached_40:,} ({reached_40/total*100:.2f}%)')

Players who reached Level 30: 33,268 (36.89%)
Players who reached Level 40: 27,392 (30.37%)


### Player Segmentation by Engagement

To enable more granular analysis, we segment players into three engagement tiers based on total rounds played. 
These thresholds are informed by the distribution observed earlier and by common industry conventions for casual mobile games

| Segment | Rounds played | Profile |
|---|---|---|
| Casual | 0 – 10 | Installed and briefly tried the game |
| Regular | 11 – 50 | Moderate engagement; approaching the gate range |
| Hardcore | 51+ | High engagement; most likely to interact with the gate |

In [13]:
# Assign engagement segments based on rounds played
# Thresholds are exploratory and should be revisited with domain input
def assign_segment(rounds):
    if rounds <= 10:
        return 'Casual'
    elif rounds <= 50:
        return 'Regular'
    else:
        return 'Hardcore'

df['segment'] = df['sum_gamerounds'].apply(assign_segment)

# Segment distribution
segment_dist = df['segment'].value_counts().reset_index()
segment_dist.columns = ['segment', 'count']
segment_dist['share (%)'] = (segment_dist['count'] / len(df) * 100).round(2)
print(segment_dist)

    segment  count  share (%)
0    Casual  35989      39.90
1   Regular  31388      34.80
2  Hardcore  22811      25.29


In [14]:
# Visualize segment distribution as a pie chart
SEGMENT_COLORS = {'Casual': '#7C3AED', 'Regular': '#F59E0B', 'Hardcore': '#10B981'}
SEGMENT_ORDER = ['Hardcore', 'Regular', 'Casual']  # ascending order for readability

fig = px.bar(
    segment_dist.set_index('segment').reindex(SEGMENT_ORDER).reset_index(),
    x='count',
    y='segment',
    color='segment',
    color_discrete_map=SEGMENT_COLORS,
    orientation='h',
    text='share (%)',
    labels={'count': 'Number of Players', 'segment': 'Segment'},
    title='Player Segmentation by Engagement Level'
)
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.update_layout(showlegend=False, xaxis_range=[0, segment_dist['count'].max() * 1.15])
fig.show()

The majority of players fall into the Casual segment, consistent with the right-skewed distribution observed earlier. 
Hardcore players represent a small but strategically important cohort, as they are most likely to encounter (and be affected by) the gate

### Retention by Segment and A/B Group

We disaggregate retention rates by both player segment and A/B group to assess whether the gate position has a differential effect across engagement tiers. 
This analysis can reveal whether the treatment disproportionately impacts casual or hardcore players

In [15]:
# Retention rates by segment and A/B group
segment_retention = (
    df.groupby(['segment', 'version'])[['retention_1', 'retention_7']]
    .mean()
    .mul(100)
    .round(2)
    .reset_index()
)
print(segment_retention)

    segment  version  retention_1  retention_7
0    Casual  gate_30        11.34         1.81
1    Casual  gate_40        11.34         1.79
2  Hardcore  gate_30        86.15        56.31
3  Hardcore  gate_40        85.10        53.74
4   Regular  gate_30        53.09        12.03
5   Regular  gate_40        52.31        10.81


In [16]:
# Side-by-side bar charts: Day-1 and Day-7 retention per segment and A/B group
segment_order = ['Casual', 'Regular', 'Hardcore']

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Day-1 Retention by Segment', 'Day-7 Retention by Segment'],
    shared_yaxes=False
)

for version in ['gate_30', 'gate_40']:
    subset = (
        segment_retention[segment_retention['version'] == version]
        .set_index('segment')
        .reindex(segment_order)
        .reset_index()
    )
    fig.add_trace(
        go.Bar(name=version, x=subset['segment'], y=subset['retention_1'],
               marker_color=COLOR_MAP[version], showlegend=True),
        row=1, col=1
    )
    fig.add_trace(
        go.Bar(name=version, x=subset['segment'], y=subset['retention_7'],
               marker_color=COLOR_MAP[version], showlegend=False),
        row=1, col=2
    )

fig.update_layout(
    barmode='group',
    title_text='Retention Rates by Player Segment and A/B Group',
    yaxis_title='Retention Rate (%)',
    yaxis2_title='Retention Rate (%)'
)
fig.show()

The retention gap between groups appears more pronounced among Regular and Hardcore players (those who engage beyond the first few rounds).
This pattern is consistent with the hypothesis that the gate position primarily affects players who progress far enough to encounter it. 
The statistical significance of these differences will be evaluated in the next notebook

### Conditional Retention: Players Who Reached the Gate

The overall retention metrics include players who never reached either gate, which dilutes the measured effect of the experiment. 
To isolate the treatment impact more precisely, we recalculate retention rates considering only players who progressed far enough to encounter the gate assigned to their group.

This conditional analysis reflects a more realistic view of the gate's influence on player behavior

In [17]:
# Filter each group to include only players who reached their respective gate
reached_gate = df[
    ((df['version'] == 'gate_30') & (df['sum_gamerounds'] >= 30)) |
    ((df['version'] == 'gate_40') & (df['sum_gamerounds'] >= 40))
]

cond_retention = (
    reached_gate.groupby('version')[['retention_1', 'retention_7']]
    .mean()
    .mul(100)
    .round(2)
    .reset_index()
)

print(f'Players who reached their gate: {len(reached_gate):,} ({len(reached_gate)/len(df)*100:.1f}% of total)')
print()
print('--- Conditional Retention Rates (%) ---')
print(cond_retention)

Players who reached their gate: 30,482 (33.8% of total)

--- Conditional Retention Rates (%) ---
   version  retention_1  retention_7
0  gate_30        80.10        43.87
1  gate_40        83.02        48.51


In [18]:
# Compare overall vs conditional retention side by side
overall = (
    df.groupby('version')[['retention_1', 'retention_7']]
    .mean()
    .mul(100)
    .round(2)
    .reset_index()
)
overall['scope'] = 'All players'
cond_retention['scope'] = 'Players who reached gate'

comparison = pd.concat([overall, cond_retention])
comparison_melted = comparison.melt(
    id_vars=['version', 'scope'],
    value_vars=['retention_1', 'retention_7'],
    var_name='metric',
    value_name='retention_rate'
)
comparison_melted['metric'] = comparison_melted['metric'].map({
    'retention_1': 'Day-1', 'retention_7': 'Day-7'
})
comparison_melted['label'] = comparison_melted['version'] + ' | ' + comparison_melted['scope']

fig = px.bar(
    comparison_melted,
    x='metric',
    y='retention_rate',
    color='version',
    facet_col='scope',
    barmode='group',
    color_discrete_map=COLOR_MAP,
    text='retention_rate',
    labels={'retention_rate': 'Retention Rate (%)', 'metric': 'Retention Window', 'version': 'A/B Group'},
    title='Overall vs. Conditional Retention (Players Who Reached the Gate)'
)
fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
fig.update_layout(yaxis_range=[0, comparison_melted['retention_rate'].max() * 1.2])
fig.show()

Among players who reached the gate, the retention difference between groups is more pronounced than in the overall population. 
This suggests that the gate position has a meaningful effect on the players directly exposed to it, and that overall metrics may underestimate the true impact of the experimental treatment

---

### **Retention Baseline Definition**

We now establish baseline retention metrics for Day 1 and Day 7, disaggregated by A/B group. These figures will serve as the primary outcome variables in the subsequent hypothesis testing notebook

In [19]:
# Overall retention rates across the full player base
overall_ret_1 = df['retention_1'].mean() * 100
overall_ret_7 = df['retention_7'].mean() * 100

print(f'Overall Day-1 Retention: {overall_ret_1:.2f}%')
print(f'Overall Day-7 Retention: {overall_ret_7:.2f}%')

Overall Day-1 Retention: 44.52%
Overall Day-7 Retention: 18.61%


In [20]:
# Retention rates disaggregated by A/B group
retention_by_version = df.groupby('version')[['retention_1', 'retention_7']].mean() * 100

print('--- Retention Rates per Version (%) ---')
print(retention_by_version.round(2))

--- Retention Rates per Version (%) ---
         retention_1  retention_7
version                          
gate_30        44.82        19.02
gate_40        44.23        18.20


In [21]:
# Grouped bar chart comparing Day-1 and Day-7 retention across A/B groups
plot_retention = retention_by_version.reset_index().melt(
    id_vars='version',
    var_name='retention_type',
    value_name='retention_rate'
)
plot_retention['retention_type'] = plot_retention['retention_type'].map({
    'retention_1': 'Day-1 Retention',
    'retention_7': 'Day-7 Retention'
})
plot_retention['retention_rate'] = plot_retention['retention_rate'].round(2)

fig = px.bar(
    plot_retention,
    x='retention_type',
    y='retention_rate',
    color='version',
    barmode='group',
    color_discrete_map=COLOR_MAP,
    text='retention_rate',
    labels={'retention_rate': 'Retention Rate (%)', 'retention_type': '', 'version': 'A/B Group'},
    title='Day-1 and Day-7 Retention Rates by A/B Group'
)
fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig.update_layout(yaxis_range=[0, plot_retention['retention_rate'].max() * 1.2])
fig.show()

Day-1 retention is nearly identical between the two groups. However, Day-7 retention appears slightly higher for `gate_30`, suggesting that the earlier gate position may contribute to better long-term engagement. Whether this difference is statistically significant will be assessed in the next notebook

---

### **Key Findings**

| Dimension | Finding |
|---|---|
| Dataset completeness | 90,189 players, zero missing values, no duplicate rows |
| Group balance | ~50/50 split confirmed - randomization appears valid |
| Engagement distribution | Heavily right-skewed; median of 16 rounds vs. mean of ~51 |
| Outlier | One extreme observation (49,854 rounds) removed - likely non-organic |
| Non-starters | 4.4% of players never engaged (0 rounds) - within normal range |
| Gate reach | Only ~34% of players reached level 30; ~28% reached level 40 |
| Day-1 retention | Nearly equivalent across both groups |
| Day-7 retention | Slightly higher for gate_30 - to be tested for significance |

**Next step:** `02_ab_test.ipynb` - frequentist hypothesis testing (bootstrap, z-test, chi-square, effect size via Cohen's h)

---

### **Export Cleaned Dataset**

In [22]:
# Export the cleaned dataset to the processed folder for use in downstream notebooks
# The only modification applied is the removal of the single extreme outlier
df.to_csv('../data/processed/cookie_cats_cleaned.csv', index=False)

print(f'Cleaned dataset exported: {df.shape[0]:,} rows, {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')

Cleaned dataset exported: 90,188 rows, 6 columns
Columns: ['userid', 'version', 'sum_gamerounds', 'retention_1', 'retention_7', 'segment']
